In [ ]:
!pip install catboost lightgbm xgboost shap

In [ ]:
# ============================================================================
# BLOCK 1: [Environment Initialization & Dependency Ingestion]
# ============================================================================
# Import all required libraries, set global random seeds for reproducibility,
# configure matplotlib styling, and suppress noisy warnings.
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os
import random

from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.model_selection import RandomizedSearchCV, learning_curve
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import skew, kurtosis

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import shap

# ── Global Seeds ──
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# ── Matplotlib Style ──
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 120,
})

# ── Output Directories ──
os.makedirs('plots', exist_ok=True)

# ── Suppress Warnings ──
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully.")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   TensorFlow: {tf.__version__}")
print(f"   Seed locked to: {SEED}")



In [ ]:
# ============================================================================
# BLOCK 2: [Anomaly-Resilient Data Ingestion Engine]
# ============================================================================
# Load train/val/test CSVs with on_bad_lines='skip' to reject malformed rows,
# then apply a multi-stage cleaning pipeline that handles:
#   1. Turkish text artifact (train.csv line 1285)
#   2. Double-comma 11-field rows (val.csv)
#   3. Swapped Age/Gender columns (val.csv, test.csv)
#   4. 11-field rows with repeated Gender (val.csv)
#   5. Missing User_ID 9-field rows (val.csv)
#   6. "Marie" gender typo (test.csv)
# Fill remaining NaNs using TRAINING-ONLY statistics to prevent data leakage.
# ============================================================================

# ── Schema Constants ──
VALID_GENDERS   = {'Male', 'Female', 'Non-binary'}
VALID_PLATFORMS = {'Instagram', 'Twitter', 'Facebook', 'LinkedIn',
                   'Snapchat', 'Telegram', 'Whatsapp'}
NUMERIC_COLS    = ['Age', 'Daily_Usage_Time (minutes)', 'Posts_Per_Day',
                   'Likes_Received_Per_Day', 'Comments_Received_Per_Day',
                   'Messages_Sent_Per_Day']
CAT_COLS        = ['Gender', 'Platform']
TARGET_COL      = 'Dominant_Emotion'

# ── Dynamic Path Resolution (Local + Google Colab compatible) ──
# In Colab the CWD is /content/; datasets may be uploaded to /content/datasets/
# or mounted via Google Drive. This block auto-detects the correct path.
DATA_DIR = 'datasets'
if not os.path.isdir(DATA_DIR):
    # Colab fallback: check if CSVs are in the current directory itself
    if os.path.isfile('train.csv'):
        DATA_DIR = '.'
    else:
        raise FileNotFoundError(
            f"Cannot find data directory. Expected 'datasets/' folder or CSV files "
            f"in the current working directory ({os.getcwd()}). "
            f"In Google Colab, upload the files first or mount your Google Drive."
        )

print(f"📂 Data directory resolved to: {os.path.abspath(DATA_DIR)}")

# ── Load Raw Data ──
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'), on_bad_lines='skip', skip_blank_lines=True)
val_raw   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'),   on_bad_lines='skip', skip_blank_lines=True)
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'),  on_bad_lines='skip', skip_blank_lines=True)

print(f"Raw shapes → Train: {train_raw.shape}, Val: {val_raw.shape}, Test: {test_raw.shape}")


def clean_dataframe(df, name):
    """
    Multi-stage anomaly-resilient cleaning pipeline.
    Handles all 6 identified anomaly types in the raw CSVs.
    """
    initial_count = len(df)

    # Stage 1: Coerce numeric columns → NaN for non-numeric values
    #          This catches swapped Age/Gender rows (e.g., "Female" in Age col)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Stage 2: Filter to valid Gender values only
    #          Catches "Marie" typo, numeric strings from swapped rows
    if 'Gender' in df.columns:
        df = df[df['Gender'].isin(VALID_GENDERS)]

    # Stage 3: Filter to valid Platform values only
    if 'Platform' in df.columns:
        df = df[df['Platform'].isin(VALID_PLATFORMS)]

    # Stage 3.5: Fix target label typos & filter unknown emotions
    #            val.csv line 121 contains 'Agression' (misspelling of 'Anger')
    LABEL_TYPO_MAP = {
        'Agression': 'Anger',
        'Aggression': 'Anger',   # guard against other common misspellings
        'Happines':  'Happiness',
        'Bordom':    'Boredom',
    }
    VALID_EMOTIONS = {'Happiness', 'Neutral', 'Anxiety', 'Sadness', 'Boredom', 'Anger'}

    if TARGET_COL in df.columns:
        df[TARGET_COL] = df[TARGET_COL].str.strip().replace(LABEL_TYPO_MAP)
        df = df[df[TARGET_COL].isin(VALID_EMOTIONS)]

    # Stage 4: Drop rows missing critical fields
    df = df.dropna(subset=['Age', 'Gender', TARGET_COL])

    # Stage 5: Drop the User_ID anchor column (not a predictive feature)
    if 'User_ID' in df.columns:
        df = df.drop(columns=['User_ID'])

    cleaned_count = len(df)
    removed = initial_count - cleaned_count
    print(f"  {name}: {initial_count} → {cleaned_count} rows  ({removed} anomalous rows removed)")
    return df.reset_index(drop=True)


train_df = clean_dataframe(train_raw.copy(), "Train")
val_df   = clean_dataframe(val_raw.copy(),   "Val  ")
test_df  = clean_dataframe(test_raw.copy(),  "Test ")

# ── Impute Remaining NaNs Using TRAINING Statistics Only ──
train_medians = train_df[NUMERIC_COLS].median()
train_modes   = {col: train_df[col].mode()[0] for col in CAT_COLS}

for df in [train_df, val_df, test_df]:
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(train_medians[col])
    for col in CAT_COLS:
        df[col] = df[col].fillna(train_modes[col])

print(f"\n✅ Cleaned shapes → Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
print(f"   Remaining NaNs → Train: {train_df.isnull().sum().sum()}, "
      f"Val: {val_df.isnull().sum().sum()}, Test: {test_df.isnull().sum().sum()}")
print(f"\n   Columns: {list(train_df.columns)}")



In [ ]:
# ============================================================================
# BLOCK 3: [Class Distribution Analysis & Imbalance Diagnostics]
# ============================================================================
# Visualize target class frequencies with bar + pie charts for the training
# set, and overlay all 3 splits to verify whether the provided partitions
# are stratified.
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
palette = sns.color_palette('viridis', n_colors=6)

# ── (a) Normalized Bar Chart — Training Set ──
emotion_counts = train_df[TARGET_COL].value_counts(normalize=True).sort_index()
emotion_counts.plot(kind='bar', ax=axes[0], color=palette, edgecolor='black', alpha=0.85)
axes[0].set_title('Training Set — Emotion Frequency', fontweight='bold')
axes[0].set_ylabel('Proportion')
axes[0].set_xlabel('')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
for i, (idx, val) in enumerate(emotion_counts.items()):
    axes[0].text(i, val + 0.005, f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')

# ── (b) Pie Chart — Training Set ──
train_df[TARGET_COL].value_counts().sort_index().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%', colors=palette,
    startangle=90, textprops={'fontsize': 9}
)
axes[1].set_title('Training Set — Proportions', fontweight='bold')
axes[1].set_ylabel('')

# ── (c) Cross-Split Comparison ──
split_dist = pd.DataFrame({
    'Train': train_df[TARGET_COL].value_counts(normalize=True),
    'Val':   val_df[TARGET_COL].value_counts(normalize=True),
    'Test':  test_df[TARGET_COL].value_counts(normalize=True),
}).sort_index()
split_dist.plot(kind='bar', ax=axes[2], alpha=0.8, edgecolor='black', width=0.75)
axes[2].set_title('Cross-Split Class Distribution', fontweight='bold')
axes[2].set_ylabel('Proportion')
axes[2].set_xlabel('')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')
axes[2].legend(title='Split', frameon=True)

plt.tight_layout()
plt.savefig('plots/block3_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Exact Training Class Counts:")
print(train_df[TARGET_COL].value_counts().sort_index().to_string())



In [ ]:
# ============================================================================
# BLOCK 4: [Continuous Feature Shift Analysis by Emotion]
# ============================================================================
# Plot overlapping Kernel Density Estimate (KDE) charts for all continuous
# features, conditioned on the Dominant_Emotion target, to identify which
# numerical features show clear distributional shifts across emotions.
# ============================================================================

kde_features = ['Daily_Usage_Time (minutes)', 'Likes_Received_Per_Day',
                'Messages_Sent_Per_Day', 'Comments_Received_Per_Day']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()

for i, feat in enumerate(kde_features):
    for emotion in sorted(train_df[TARGET_COL].unique()):
        subset = train_df[train_df[TARGET_COL] == emotion][feat]
        subset.plot(kind='kde', ax=axes_flat[i], label=emotion, alpha=0.5, linewidth=2)
    axes_flat[i].set_title(f'KDE — {feat}', fontweight='bold')
    axes_flat[i].set_xlabel(feat)
    axes_flat[i].legend(fontsize=8, title='Emotion')
    axes_flat[i].set_ylabel('Density')

plt.suptitle('Continuous Feature Distributions Conditioned on Emotion', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/block4_kde_shifts.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# BLOCK 5: [Categorical Cross-Tabulation & Platform Behavior Matrix]
# ============================================================================
# Evaluate how Dominant_Emotion varies across Platform and Gender using
# stacked bar charts and grouped bar charts.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# ── (a) Platform × Emotion — Stacked ──
platform_emotion = pd.crosstab(train_df['Platform'], train_df[TARGET_COL], normalize='index')
platform_emotion.plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis',
                      edgecolor='black', alpha=0.85)
axes[0].set_title('Platform × Emotion (Normalized, Stacked)', fontweight='bold')
axes[0].set_ylabel('Proportion')
axes[0].set_xlabel('')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(title='Emotion', fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')

# ── (b) Gender × Emotion — Grouped ──
gender_emotion = pd.crosstab(train_df['Gender'], train_df[TARGET_COL])
gender_emotion.plot(kind='bar', ax=axes[1], colormap='viridis',
                    edgecolor='black', alpha=0.85)
axes[1].set_title('Gender × Emotion (Absolute Count)', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Emotion', fontsize=8)

plt.tight_layout()
plt.savefig('plots/block5_categorical_crosstab.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# ============================================================================
# BLOCK 6: [Bivariate Separability Scatter Space]
# ============================================================================
# Investigate pairwise feature interactions using scatter plots colored by
# the target class label to see if clusters emerge visually.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── (a) Daily Usage Time vs Messages Sent ──
for emotion in sorted(train_df[TARGET_COL].unique()):
    subset = train_df[train_df[TARGET_COL] == emotion]
    axes[0].scatter(subset['Daily_Usage_Time (minutes)'], subset['Messages_Sent_Per_Day'],
                    label=emotion, alpha=0.5, s=40, edgecolors='k', linewidth=0.3)
axes[0].set_xlabel('Daily Usage Time (minutes)')
axes[0].set_ylabel('Messages Sent Per Day')
axes[0].set_title('Usage Time vs Messages — by Emotion', fontweight='bold')
axes[0].legend(fontsize=8, title='Emotion')

# ── (b) Likes Received vs Comments Received ──
for emotion in sorted(train_df[TARGET_COL].unique()):
    subset = train_df[train_df[TARGET_COL] == emotion]
    axes[1].scatter(subset['Likes_Received_Per_Day'], subset['Comments_Received_Per_Day'],
                    label=emotion, alpha=0.5, s=40, edgecolors='k', linewidth=0.3)
axes[1].set_xlabel('Likes Received Per Day')
axes[1].set_ylabel('Comments Received Per Day')
axes[1].set_title('Likes vs Comments — by Emotion', fontweight='bold')
axes[1].legend(fontsize=8, title='Emotion')

plt.tight_layout()
plt.savefig('plots/block6_scatter_separability.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# BLOCK 7: [Distribution Profile: Skewness, Kurtosis & Outlier Detection]
# ============================================================================
# Compute Fisher-Pearson skewness and excess kurtosis for all numerical
# features, then visualize box + violin plots partitioned by emotion.
# ============================================================================

# ── Statistical Moments Table ──
stats_table = pd.DataFrame({
    'Feature': NUMERIC_COLS,
    'Skewness':       [skew(train_df[col].dropna())     for col in NUMERIC_COLS],
    'Excess Kurtosis': [kurtosis(train_df[col].dropna()) for col in NUMERIC_COLS],
    'Mean':            [train_df[col].mean()              for col in NUMERIC_COLS],
    'Std':             [train_df[col].std()               for col in NUMERIC_COLS],
})
print("📐 Statistical Moments — Training Set")
print(stats_table.to_string(index=False))

# ── Box Plots ──
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes_flat = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    sns.boxplot(data=train_df, x=TARGET_COL, y=col, ax=axes_flat[i],
                palette='viridis', order=sorted(train_df[TARGET_COL].unique()))
    axes_flat[i].set_title(f'{col}', fontweight='bold')
    axes_flat[i].set_xlabel('')
    axes_flat[i].tick_params(axis='x', rotation=45)

plt.suptitle('Box Plots by Dominant Emotion', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/block7_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Violin Plots ──
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes_flat = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    sns.violinplot(data=train_df, x=TARGET_COL, y=col, ax=axes_flat[i],
                   palette='mako', inner='quartile',
                   order=sorted(train_df[TARGET_COL].unique()))
    axes_flat[i].set_title(f'{col}', fontweight='bold')
    axes_flat[i].set_xlabel('')
    axes_flat[i].tick_params(axis='x', rotation=45)

plt.suptitle('Violin Plots by Dominant Emotion', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/block7_violinplots.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# ============================================================================
# BLOCK 8: [Unsupervised PCA Variance Clustering]
# ============================================================================
# Standardize continuous features and project them into a 2-component PCA
# space. Scatter the principal components, colored by emotion, to evaluate
# whether natural clusters emerge without supervised guidance.
# ============================================================================

pca_scaler = StandardScaler()
train_num_scaled = pca_scaler.fit_transform(train_df[NUMERIC_COLS])

pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(train_num_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
for emotion in sorted(train_df[TARGET_COL].unique()):
    mask = train_df[TARGET_COL] == emotion
    ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
               label=emotion, alpha=0.55, s=50, edgecolors='k', linewidth=0.3)

ax.set_xlabel(f'PC-1  ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC-2  ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA 2-Component Projection — Colored by Emotion', fontweight='bold')
ax.legend(title='Emotion', fontsize=9)
plt.tight_layout()
plt.savefig('plots/block8_pca_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"📐 Explained Variance Ratio: PC-1 = {pca.explained_variance_ratio_[0]:.4f}, "
      f"PC-2 = {pca.explained_variance_ratio_[1]:.4f}")
print(f"   Total variance captured: {pca.explained_variance_ratio_.sum():.4f}")



In [ ]:
# ============================================================================
# BLOCK 9: [Advanced Feature Engineering Pipeline]
# ============================================================================
# Compute 6 derived behavioral interaction features across all partitions.
# These capture engagement intensity, content efficiency, and conversational
# patterns that raw counts alone cannot represent.
# ============================================================================

EPS = 1e-5   # epsilon to prevent division by zero

def engineer_features(df):
    """Create 6 advanced behavioral interaction features."""
    df = df.copy()
    df['Interaction_Density']        = (df['Likes_Received_Per_Day'] + df['Comments_Received_Per_Day']) / (df['Daily_Usage_Time (minutes)'] + EPS)
    df['Social_Velocity']            = df['Likes_Received_Per_Day'] / (df['Posts_Per_Day'] + EPS)
    df['Conversational_Reciprocity'] = df['Messages_Sent_Per_Day'] / (df['Comments_Received_Per_Day'] + EPS)
    df['Attention_Index']            = df['Daily_Usage_Time (minutes)'] / (df['Posts_Per_Day'] + EPS)
    df['Engagement_Ratio']           = (df['Likes_Received_Per_Day'] + df['Comments_Received_Per_Day'] + df['Messages_Sent_Per_Day']) / (df['Daily_Usage_Time (minutes)'] + EPS)
    df['Content_Efficiency']         = df['Likes_Received_Per_Day'] / (df['Daily_Usage_Time (minutes)'] * df['Posts_Per_Day'] + EPS)
    return df

train_df = engineer_features(train_df)
val_df   = engineer_features(val_df)
test_df  = engineer_features(test_df)

new_feats = ['Interaction_Density', 'Social_Velocity', 'Conversational_Reciprocity',
             'Attention_Index', 'Engagement_Ratio', 'Content_Efficiency']

print("✅ 6 Engineered features added to all partitions.")
print(f"   New columns: {new_feats}")
print(f"   Train shape: {train_df.shape}, Val shape: {val_df.shape}, Test shape: {test_df.shape}")
print(f"\n   Training summary of new features:")
print(train_df[new_feats].describe().round(3).to_string())



In [ ]:
# ============================================================================
# BLOCK 10: [Categorical Encoding & Schema Alignment]
# ============================================================================
# One-hot encode Gender and Platform, then align val/test column schemas
# to match the training set exactly using reindex. Encode the target with
# LabelEncoder.
# ============================================================================

# ── Separate target from features ──
y_train = train_df[TARGET_COL].copy()
y_val   = val_df[TARGET_COL].copy()
y_test  = test_df[TARGET_COL].copy()

X_train = train_df.drop(columns=[TARGET_COL])
X_val   = val_df.drop(columns=[TARGET_COL])
X_test  = test_df.drop(columns=[TARGET_COL])

# ── One-Hot Encode categoricals ──
X_train = pd.get_dummies(X_train, columns=CAT_COLS, dtype=float)
X_val   = pd.get_dummies(X_val,   columns=CAT_COLS, dtype=float)
X_test  = pd.get_dummies(X_test,  columns=CAT_COLS, dtype=float)

# ── Align validation and test columns to training schema ──
X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# ── Encode target labels ──
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_val   = encoder.transform(y_val)
y_test  = encoder.transform(y_test)

n_classes    = len(encoder.classes_)
class_names  = list(encoder.classes_)
feature_names = list(X_train.columns)

print(f"✅ Encoding complete.")
print(f"   Feature count: {len(feature_names)}")
print(f"   Target classes ({n_classes}): {class_names}")
print(f"   Label mapping: { {cls: idx for idx, cls in enumerate(class_names)} }")
print(f"   X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")



In [ ]:
# ============================================================================
# BLOCK 11: [Multicollinearity Elimination Filter]
# ============================================================================
# Compute the absolute Pearson correlation matrix and drop any feature
# whose pairwise cross-correlation with another feature exceeds r > 0.85.
# ============================================================================

# ── Compute correlation matrix ──
corr_matrix = X_train.corr().abs()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', ax=ax, linewidths=0.5,
            xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlation Matrix (Absolute Pearson)', fontweight='bold')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('plots/block11_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Identify features to drop (upper triangle, threshold 0.85) ──
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
cols_to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

if cols_to_drop:
    print(f"⚠️  Dropping {len(cols_to_drop)} highly correlated features: {cols_to_drop}")
    X_train = X_train.drop(columns=cols_to_drop)
    X_val   = X_val.drop(columns=cols_to_drop)
    X_test  = X_test.drop(columns=cols_to_drop)
    feature_names = list(X_train.columns)
else:
    print("✅ No features exceed the r > 0.85 multicollinearity threshold.")

print(f"   Retained feature count: {len(feature_names)}")



In [ ]:
# ============================================================================
# BLOCK 12: [Partition-Isolated Feature Scaling]
# ============================================================================
# Fit StandardScaler ONLY on training features, then transform val and test
# partitions with the same fitted scaler to prevent data leakage.
# ============================================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Feature scaling complete (fit on train only).")
print(f"   X_train_scaled: {X_train_scaled.shape}")
print(f"   X_val_scaled:   {X_val_scaled.shape}")
print(f"   X_test_scaled:  {X_test_scaled.shape}")
print(f"   Train mean ≈ 0: {X_train_scaled.mean(axis=0).mean():.6f}")
print(f"   Train std  ≈ 1: {X_train_scaled.std(axis=0).mean():.6f}")


In [ ]:
# ============================================================================
# BLOCK 13: [Classical ML Baseline — Random Forest & Logistic Regression]
# ============================================================================
# Train two baseline classifiers to establish a performance floor before
# moving to advanced ensemble methods. Both use class_weight='balanced'
# to handle potential class imbalance.
# ============================================================================

# ── Storage for all models ──
trained_models = {}       # name → fitted model
model_val_f1   = {}       # name → validation F1 (for ensemble selection)

# ── Random Forest ──
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
rf_val_pred = rf_model.predict(X_val_scaled)
rf_val_f1 = f1_score(y_val, rf_val_pred, average='weighted')
trained_models['Random Forest'] = rf_model
model_val_f1['Random Forest'] = rf_val_f1
print(f"   Random Forest — Val Weighted F1: {rf_val_f1:.4f}")

# ── Logistic Regression ──
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    class_weight='balanced', max_iter=2000, multi_class='multinomial',
    solver='lbfgs', random_state=SEED
)
lr_model.fit(X_train_scaled, y_train)
lr_val_pred = lr_model.predict(X_val_scaled)
lr_val_f1 = f1_score(y_val, lr_val_pred, average='weighted')
trained_models['Logistic Regression'] = lr_model
model_val_f1['Logistic Regression'] = lr_val_f1
print(f"   Logistic Regression — Val Weighted F1: {lr_val_f1:.4f}")

print(f"\n✅ Baselines trained. RF: {rf_val_f1:.4f}, LR: {lr_val_f1:.4f}")



In [ ]:
# ============================================================================
# BLOCK 14: [Advanced Gradient Boosting — CatBoost, LightGBM, XGBoost]
# ============================================================================
# Train three SOTA gradient boosting classifiers with RandomizedSearchCV
# hyperparameter tuning. Scoring uses weighted F1 to handle multi-class
# imbalance properly.
# ============================================================================

# ── CatBoost ──
print("Tuning CatBoost...")
cat_param_grid = {
    'depth':         [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1, 0.2],
    'iterations':    [200, 400, 600],
    'l2_leaf_reg':   [1, 3, 5, 7],
}
cat_base = CatBoostClassifier(
    auto_class_weights='Balanced', random_seed=SEED,
    verbose=0, allow_writing_files=False
)
cat_search = RandomizedSearchCV(
    cat_base, cat_param_grid, n_iter=20, cv=3,
    scoring='f1_weighted', random_state=SEED, n_jobs=-1
)
cat_search.fit(X_train_scaled, y_train)
cat_model = cat_search.best_estimator_
cat_val_pred = cat_model.predict(X_val_scaled)
cat_val_f1 = f1_score(y_val, cat_val_pred, average='weighted')
trained_models['CatBoost'] = cat_model
model_val_f1['CatBoost'] = cat_val_f1
print(f"   CatBoost — Val F1: {cat_val_f1:.4f} | Best params: {cat_search.best_params_}")

# ── LightGBM ──
print("Tuning LightGBM...")
lgb_param_grid = {
    'num_leaves':       [15, 31, 50],
    'learning_rate':    [0.03, 0.05, 0.1, 0.2],
    'n_estimators':     [200, 400, 600],
    'max_depth':        [3, 5, 7, -1],
    'min_child_samples': [5, 10, 20],
}
lgb_base = LGBMClassifier(
    class_weight='balanced', random_state=SEED, verbose=-1, n_jobs=-1
)
lgb_search = RandomizedSearchCV(
    lgb_base, lgb_param_grid, n_iter=20, cv=3,
    scoring='f1_weighted', random_state=SEED, n_jobs=-1
)
lgb_search.fit(X_train_scaled, y_train)
lgb_model = lgb_search.best_estimator_
lgb_val_pred = lgb_model.predict(X_val_scaled)
lgb_val_f1 = f1_score(y_val, lgb_val_pred, average='weighted')
trained_models['LightGBM'] = lgb_model
model_val_f1['LightGBM'] = lgb_val_f1
print(f"   LightGBM — Val F1: {lgb_val_f1:.4f} | Best params: {lgb_search.best_params_}")

# ── XGBoost ──
print("Tuning XGBoost...")
xgb_param_grid = {
    'max_depth':        [3, 5, 7, 9],
    'learning_rate':    [0.03, 0.05, 0.1, 0.2],
    'n_estimators':     [200, 400, 600],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}
xgb_base = XGBClassifier(
    objective='multi:softprob', eval_metric='mlogloss',
    random_state=SEED, n_jobs=-1, verbosity=0
)
# Compute balanced sample weights for XGBoost (no class_weight param)
sample_weights_train = compute_sample_weight('balanced', y_train)

xgb_search = RandomizedSearchCV(
    xgb_base, xgb_param_grid, n_iter=20, cv=3,
    scoring='f1_weighted', random_state=SEED, n_jobs=-1
)
xgb_search.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)
xgb_model = xgb_search.best_estimator_
xgb_val_pred = xgb_model.predict(X_val_scaled)
xgb_val_f1 = f1_score(y_val, xgb_val_pred, average='weighted')
trained_models['XGBoost'] = xgb_model
model_val_f1['XGBoost'] = xgb_val_f1
print(f"   XGBoost — Val F1: {xgb_val_f1:.4f} | Best params: {xgb_search.best_params_}")

print(f"\n✅ Gradient Boosting models tuned.")
print(f"   CatBoost: {cat_val_f1:.4f}, LightGBM: {lgb_val_f1:.4f}, XGBoost: {xgb_val_f1:.4f}")


In [ ]:
# ============================================================================
# BLOCK 15: [Deep Learning Model 1 — Multi-Layer Perceptron (MLP)]
# ============================================================================
# Construct a 3-hidden-layer feedforward network with BatchNormalization,
# Dropout regularization, and EarlyStopping + ReduceLROnPlateau callbacks.
# ============================================================================

input_dim = X_train_scaled.shape[1]

mlp_model = Sequential([
    Dense(256, activation='relu', input_shape=(input_dim,)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(n_classes, activation='softmax')
], name='MLP_Classifier')

mlp_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mlp_callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("Training MLP...")
mlp_history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200, batch_size=32,
    callbacks=mlp_callbacks, verbose=0
)

mlp_val_probs = mlp_model.predict(X_val_scaled, verbose=0)
mlp_val_pred  = np.argmax(mlp_val_probs, axis=1)
mlp_val_f1    = f1_score(y_val, mlp_val_pred, average='weighted')
trained_models['MLP'] = mlp_model
model_val_f1['MLP']  = mlp_val_f1
print(f"✅ MLP — Val Weighted F1: {mlp_val_f1:.4f}")
mlp_model.summary()

In [ ]:
# ============================================================================
# BLOCK 16: [Deep Learning Model 2 — Swish-Activated Deep Network]
# ============================================================================
# Build an alternative deeper network using Swish (SiLU) activations and
# the RMSprop optimizer to explore different convergence dynamics.
# ============================================================================

swish_model = Sequential([
    Dense(512, activation='swish', input_shape=(input_dim,)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(256, activation='swish'),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='swish'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='swish'),
    Dropout(0.2),

    Dense(n_classes, activation='softmax')
], name='Swish_Deep_Net')

swish_model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

swish_callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("Training Swish Deep Net...")
swish_history = swish_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200, batch_size=32,
    callbacks=swish_callbacks, verbose=0
)

swish_val_probs = swish_model.predict(X_val_scaled, verbose=0)
swish_val_pred  = np.argmax(swish_val_probs, axis=1)
swish_val_f1    = f1_score(y_val, swish_val_pred, average='weighted')
trained_models['Swish-Net'] = swish_model
model_val_f1['Swish-Net']  = swish_val_f1
print(f"✅ Swish-Net — Val Weighted F1: {swish_val_f1:.4f}")
swish_model.summary()



In [ ]:
# ============================================================================
# BLOCK 17: [Training History Visualization]
# ============================================================================
# Plot loss and accuracy curves for both DL models (train vs validation)
# to diagnose convergence speed and overfitting.
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

histories = {'MLP': mlp_history, 'Swish-Net': swish_history}

for i, (name, hist) in enumerate(histories.items()):
    # ── Loss Curve ──
    axes[0, i].plot(hist.history['loss'], label='Train Loss', linewidth=2)
    axes[0, i].plot(hist.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
    axes[0, i].set_title(f'{name} — Loss Curve', fontweight='bold')
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].set_ylabel('Loss')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)

    # ── Accuracy Curve ──
    axes[1, i].plot(hist.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[1, i].plot(hist.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
    axes[1, i].set_title(f'{name} — Accuracy Curve', fontweight='bold')
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].set_ylabel('Accuracy')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Deep Learning Training History', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/block17_training_history.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# BLOCK 18: [Soft-Voting Ensemble of Top Models]
# ============================================================================
# Select the top 3 sklearn-compatible models by validation F1, and create
# a manual soft-voting ensemble that averages their predicted probabilities.
# ============================================================================

# ── Identify top 3 sklearn models (exclude Keras models) ──
sklearn_models = {k: v for k, v in model_val_f1.items() if k not in ('MLP', 'Swish-Net')}
top3_names = sorted(sklearn_models, key=sklearn_models.get, reverse=True)[:3]

print(f"🏆 Top 3 sklearn models for ensemble: {top3_names}")
for name in top3_names:
    print(f"   {name}: Val F1 = {model_val_f1[name]:.4f}")

# ── Compute ensemble predictions on validation set ──
ensemble_val_probs = np.mean(
    [trained_models[name].predict_proba(X_val_scaled) for name in top3_names],
    axis=0
)
ensemble_val_pred = np.argmax(ensemble_val_probs, axis=1)
ensemble_val_f1   = f1_score(y_val, ensemble_val_pred, average='weighted')

model_val_f1['Soft-Vote Ensemble'] = ensemble_val_f1
print(f"\n✅ Soft-Vote Ensemble — Val Weighted F1: {ensemble_val_f1:.4f}")


In [ ]:
# ============================================================================
# BLOCK 19: [Comprehensive Model Evaluation Matrix]
# ============================================================================
# Evaluate ALL models on the held-out TEST set. Compute weighted AND macro
# Accuracy, Precision, Recall, F1-Score. Rank by weighted F1 to crown the
# Champion Model.
# ============================================================================

results = []

for name, model in trained_models.items():
    # ── Generate test predictions ──
    if name in ('MLP', 'Swish-Net'):
        probs = model.predict(X_test_scaled, verbose=0)
        preds = np.argmax(probs, axis=1)
    else:
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)

    results.append({
        'Model': name,
        'Accuracy':          accuracy_score(y_test, preds),
        'Precision (W)':     precision_score(y_test, preds, average='weighted', zero_division=0),
        'Recall (W)':        recall_score(y_test, preds, average='weighted', zero_division=0),
        'F1-Score (W)':      f1_score(y_test, preds, average='weighted', zero_division=0),
        'Precision (Macro)': precision_score(y_test, preds, average='macro', zero_division=0),
        'Recall (Macro)':    recall_score(y_test, preds, average='macro', zero_division=0),
        'F1-Score (Macro)':  f1_score(y_test, preds, average='macro', zero_division=0),
    })

# ── Ensemble test evaluation ──
ensemble_test_probs = np.mean(
    [trained_models[name].predict_proba(X_test_scaled) for name in top3_names],
    axis=0
)
ensemble_test_pred = np.argmax(ensemble_test_probs, axis=1)
results.append({
    'Model': 'Soft-Vote Ensemble',
    'Accuracy':          accuracy_score(y_test, ensemble_test_pred),
    'Precision (W)':     precision_score(y_test, ensemble_test_pred, average='weighted', zero_division=0),
    'Recall (W)':        recall_score(y_test, ensemble_test_pred, average='weighted', zero_division=0),
    'F1-Score (W)':      f1_score(y_test, ensemble_test_pred, average='weighted', zero_division=0),
    'Precision (Macro)': precision_score(y_test, ensemble_test_pred, average='macro', zero_division=0),
    'Recall (Macro)':    recall_score(y_test, ensemble_test_pred, average='macro', zero_division=0),
    'F1-Score (Macro)':  f1_score(y_test, ensemble_test_pred, average='macro', zero_division=0),
})

results_df = pd.DataFrame(results).sort_values('F1-Score (W)', ascending=False).reset_index(drop=True)

print("\n" + "=" * 100)
print("🏆 COMPREHENSIVE MODEL PERFORMANCE MATRIX — TEST SET")
print("=" * 100)
print(results_df.to_string(index=False, float_format='%.4f'))

champion_name = results_df.iloc[0]['Model']
print(f"\n🥇 CHAMPION MODEL: {champion_name} (Test F1-W: {results_df.iloc[0]['F1-Score (W)']:.4f})")


In [ ]:
# ============================================================================
# BLOCK 20: [Per-Class Classification Report]
# ============================================================================
# Generate a detailed per-class precision / recall / F1 breakdown for the
# champion model to identify which specific emotions it handles well or
# struggles with.
# ============================================================================

# ── Get champion predictions ──
if champion_name == 'Soft-Vote Ensemble':
    champion_test_pred  = ensemble_test_pred
    champion_test_probs = ensemble_test_probs
elif champion_name in ('MLP', 'Swish-Net'):
    champion_test_probs = trained_models[champion_name].predict(X_test_scaled, verbose=0)
    champion_test_pred  = np.argmax(champion_test_probs, axis=1)
else:
    champion_test_pred  = trained_models[champion_name].predict(X_test_scaled)
    champion_test_probs = trained_models[champion_name].predict_proba(X_test_scaled)

print(f"\n📋 Per-Class Classification Report — {champion_name}")
print("=" * 70)
print(classification_report(y_test, champion_test_pred,
                            target_names=class_names, digits=4, zero_division=0))


In [ ]:
# ============================================================================
# BLOCK 21: [ROC-AUC & Precision-Recall Curves]
# ============================================================================
# Compute One-vs-Rest ROC curves and Precision-Recall curves for all 6
# emotion classes using the champion model's predicted probabilities.
# ============================================================================

y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
colors = plt.cm.viridis(np.linspace(0, 0.9, n_classes))

# ── ROC Curves ──
for i, cls_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], champion_test_probs[:, i])
    roc_auc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=colors[i], linewidth=2,
                 label=f'{cls_name} (AUC = {roc_auc_val:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curves (OvR) — {champion_name}', fontweight='bold')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(True, alpha=0.3)

# ── Precision-Recall Curves ──
for i, cls_name in enumerate(class_names):
    prec, rec, _ = precision_recall_curve(y_test_bin[:, i], champion_test_probs[:, i])
    ap = average_precision_score(y_test_bin[:, i], champion_test_probs[:, i])
    axes[1].plot(rec, prec, color=colors[i], linewidth=2,
                 label=f'{cls_name} (AP = {ap:.3f})')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curves (OvR) — {champion_name}', fontweight='bold')
axes[1].legend(fontsize=9, loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/block21_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# ============================================================================
# BLOCK 22: [Confusion Matrix & SHAP Explainability Dashboard]
# ============================================================================
# (Left) Normalized confusion matrix heatmap for the champion model.
# (Right) SHAP TreeExplainer summary plot showing feature importance per class.
# If champion is DL, SHAP uses the best tree model instead.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# ── (a) Confusion Matrix ──
cm = confusion_matrix(y_test, champion_test_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, cbar_kws={'label': 'Proportion'})
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_title(f'Normalized Confusion Matrix — {champion_name}', fontweight='bold')

# ── (b) SHAP Feature Importance ──
# Use the best tree-based model for SHAP (TreeExplainer is fast & reliable)
tree_model_names = [n for n in ['CatBoost', 'LightGBM', 'XGBoost', 'Random Forest']
                    if n in trained_models]
best_tree_name = max(tree_model_names, key=lambda n: model_val_f1.get(n, 0))
best_tree_model = trained_models[best_tree_name]

print(f"🔍 Using {best_tree_name} for SHAP explainability...")

X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

try:
    explainer = shap.TreeExplainer(best_tree_model)
    shap_values = explainer.shap_values(X_test_df)

    plt.sca(axes[1])
    # ── Safe, version-agnostic multi-class SHAP averaging ──
    # shap_values can be either:
    #   (a) a Python list of 2D arrays (one per class) — older SHAP / some models
    #   (b) a single 3D numpy array of shape (samples, features, classes) — newer SHAP
    # We must handle both to avoid a pd.Series "Data must be 1-dimensional" crash.
    if isinstance(shap_values, list):
        # Case (a): list of 2D arrays → mean across classes, then across samples
        mean_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
        final_shap_vectors = mean_abs_shap.mean(axis=0)
    else:
        shap_arr = np.array(shap_values)
        if shap_arr.ndim == 3:
            # Case (b): 3D array (samples, features, classes)
            # Average absolute values across both samples (axis=0) and classes (axis=2)
            final_shap_vectors = np.abs(shap_arr).mean(axis=(0, 2))
        else:
            # Fallback: standard 2D array (samples, features)
            final_shap_vectors = np.abs(shap_arr).mean(axis=0)

    feature_importance_shap = pd.Series(
        final_shap_vectors, index=feature_names
    ).sort_values(ascending=True)

    feature_importance_shap.plot(kind='barh', ax=axes[1], color=sns.color_palette('viridis', len(feature_names)),
                                 edgecolor='black', alpha=0.85)
    axes[1].set_title(f'SHAP Mean |Value| — {best_tree_name}', fontweight='bold')
    axes[1].set_xlabel('Mean |SHAP Value|')

except Exception as e:
    print(f"⚠️  SHAP computation failed: {e}")
    axes[1].text(0.5, 0.5, 'SHAP unavailable', transform=axes[1].transAxes,
                 ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig('plots/block22_confusion_shap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Full SHAP Summary Plot (separate figure) ──
try:
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False,
                      class_names=class_names)
    plt.title(f'SHAP Summary — {best_tree_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('plots/block22_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"⚠️  SHAP summary plot failed: {e}")


In [ ]:
# ============================================================================
# BLOCK 23: [Learning Curve Analysis]
# ============================================================================
# Generate a learning curve for the best tree model to diagnose whether the
# model is underfitting (high bias) or overfitting (high variance) as the
# training set size increases.
# ============================================================================

print(f"📈 Computing learning curve for {best_tree_name}...")

try:
    train_sizes, train_scores, val_scores = learning_curve(
        best_tree_model, X_train_scaled, y_train,
        cv=5, scoring='f1_weighted',
        train_sizes=np.linspace(0.1, 1.0, 10),
        random_state=SEED, n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='orange')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2, label='Training Score')
    ax.plot(train_sizes, val_mean,   'o-', color='orange', linewidth=2, label='CV Validation Score')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Weighted F1-Score')
    ax.set_title(f'Learning Curve — {best_tree_name}', fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('plots/block23_learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f"⚠️  Learning curve computation failed: {e}")


In [ ]:
# ============================================================================
# BLOCK 24: [Pipeline Serialization & Export]
# ============================================================================
# Serialize the champion model, scaler, encoder, and comprehensive metadata
# into persistent binary files for production deployment.
#
# DESIGN DECISION — Unified Pickle Contract:
#   All champion models (Tree, Ensemble, Keras) are wrapped in a standardized
#   ChampionModelWrapper class that always exposes .predict() and
#   .predict_proba(). This guarantees the Streamlit app never needs to
#   inspect what's inside the pickle — it just calls the same two methods.
# ============================================================================


class ChampionModelWrapper:
    """
    Unified inference wrapper that standardizes the .predict_proba() contract
    regardless of whether the champion is a single tree model, a soft-vote
    ensemble, or a Keras neural network.
    """

    def __init__(self, model_or_models, deploy_type, model_names=None):
        self.deploy_type = deploy_type
        self.model_names = model_names  # only for Ensemble
        if deploy_type == 'Ensemble':
            self._models = model_or_models  # list of sklearn estimators
        else:
            self._model = model_or_models   # single estimator

    def predict_proba(self, X):
        """Return class probability array of shape (n_samples, n_classes)."""
        if self.deploy_type == 'Ensemble':
            return np.mean([m.predict_proba(X) for m in self._models], axis=0)
        elif self.deploy_type == 'Keras':
            return self._model.predict(X, verbose=0)
        else:
            return self._model.predict_proba(X)

    def predict(self, X):
        """Return integer class predictions."""
        return np.argmax(self.predict_proba(X), axis=1)


# ── Determine champion type and build the wrapper ──
is_keras_champion = champion_name in ('MLP', 'Swish-Net')

if is_keras_champion:
    # Keras models cannot be pickled — save weights to native .keras format
    keras_model_obj = trained_models[champion_name]
    keras_model_obj.save('champion_model.keras')
    model_deploy_type = 'Keras'
    print(f"✅ Keras champion weights saved → champion_model.keras")
    # Also save a lightweight wrapper (without the Keras model inside)
    # The Streamlit app will load the .keras file separately via tf.keras
    wrapper = ChampionModelWrapper(keras_model_obj, 'Keras')
    # NOTE: wrapper is NOT pickled for Keras — app.py loads .keras directly

elif champion_name == 'Soft-Vote Ensemble':
    ensemble_models = [trained_models[n] for n in top3_names]
    wrapper = ChampionModelWrapper(ensemble_models, 'Ensemble', model_names=top3_names)
    with open('champion_model.pkl', 'wb') as f:
        pickle.dump(wrapper, f)
    model_deploy_type = 'Ensemble'
    print(f"✅ Ensemble wrapper saved → champion_model.pkl")

else:
    wrapper = ChampionModelWrapper(trained_models[champion_name], 'Tree')
    with open('champion_model.pkl', 'wb') as f:
        pickle.dump(wrapper, f)
    model_deploy_type = 'Tree'
    print(f"✅ {champion_name} wrapper saved → champion_model.pkl")

# ── Save Scaler ──
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved → scaler.pkl")

# ── Save Encoder ──
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
print("✅ Encoder saved → encoder.pkl")

# ── Compute feature importance from best tree model ──
try:
    if hasattr(best_tree_model, 'feature_importances_'):
        feat_imp = dict(zip(feature_names, best_tree_model.feature_importances_.tolist()))
    else:
        feat_imp = {}
except Exception:
    feat_imp = {}

# ── Build comprehensive metadata ──
metadata = {
    'retained_features_list':  feature_names,
    'label_encoder_classes':   class_names,
    'model_deployment_type':   model_deploy_type,
    'champion_model_name':     champion_name,
    'dropped_features':        cols_to_drop if cols_to_drop else [],
    'n_classes':               n_classes,
    'feature_importance':      feat_imp,
    'confusion_matrix':        cm.tolist(),
    'model_performance':       results_df.to_dict('records'),
    'class_distribution':      dict(pd.Series(y_train).value_counts().sort_index()),
    'ensemble_model_names':    top3_names,
    'best_tree_model_name':    best_tree_name,
}

with open('pipeline_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("✅ Metadata saved → pipeline_metadata.pkl")

# ── Also save the best tree model separately (for SHAP in the app) ──
with open('best_tree_model.pkl', 'wb') as f:
    pickle.dump(best_tree_model, f)
print(f"✅ Best tree model ({best_tree_name}) saved → best_tree_model.pkl")

print("\n" + "=" * 70)
print(f"🎯 PIPELINE COMPLETE — Champion: {champion_name}")
print(f"   Deployment type: {model_deploy_type}")
print(f"   Pickle contract: ChampionModelWrapper (always exposes .predict_proba())")
print(f"   Files saved: champion_model.{'keras' if is_keras_champion else 'pkl'}, ")
print(f"                scaler.pkl, encoder.pkl, pipeline_metadata.pkl, best_tree_model.pkl")
print("=" * 70)


In [ ]:
%%writefile requirements.txt
streamlit>=1.30.0
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
tensorflow-cpu>=2.14.0
lightgbm>=4.0.0
catboost>=1.2.0
xgboost>=1.7.0
shap>=0.42.0
matplotlib>=3.7.0
seaborn>=0.12.0
plotly>=5.18.0
scipy>=1.10.0

In [ ]:
%%writefile .python-version
3.10 %%writefile app.py

In [ ]:
%%writefile app.py
# ============================================================================
# BLOCK 25 — NeuroSense v3: Production Streamlit Dashboard
# ============================================================================
# Futuristic dark-mode analytics platform with 6 tabs.
# All critical deployment precautions are preserved:
#   1. ChampionModelWrapper defined BEFORE pickle.load()
#   2. TensorFlow imported lazily (only if champion is Keras)
#   3. Unified .predict_proba() contract — no type-sniffing
# ============================================================================

import streamlit as st
import numpy as np
import pandas as pd
import pickle
import plotly.graph_objects as go
import plotly.express as px

# ── Lazy TF — NOT imported at module level ──
TF_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False


# ============================================================================
# CHAMPION MODEL WRAPPER  (pickle needs this class in the module namespace)
# ============================================================================

class ChampionModelWrapper:
    """Unified wrapper: always exposes .predict_proba() and .predict()."""

    def __init__(self, model_or_models, deploy_type, model_names=None):
        self.deploy_type = deploy_type
        self.model_names = model_names
        if deploy_type == "Ensemble":
            self._models = model_or_models
        else:
            self._model = model_or_models

    def predict_proba(self, X):
        if self.deploy_type == "Ensemble":
            return np.mean([m.predict_proba(X) for m in self._models], axis=0)
        elif self.deploy_type == "Keras":
            return self._model.predict(X, verbose=0)
        else:
            return self._model.predict_proba(X)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


# ============================================================================
# PAGE CONFIG
# ============================================================================

st.set_page_config(
    page_title="NeuroSense — AI Emotion Analytics",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="collapsed",
)

# ============================================================================
# CSS — Futuristic Dark Neon Theme
# ============================================================================

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&family=Space+Grotesk:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap');

/* ── Root Variables ── */
:root {
    --accent-1: #7c3aed;
    --accent-2: #06d6a0;
    --accent-3: #3b82f6;
    --bg-dark: #05051a;
    --bg-card: rgba(255,255,255,0.03);
    --border: rgba(255,255,255,0.06);
    --text-primary: #e0e6ed;
    --text-secondary: #6b7280;
    --text-muted: #4b5563;
}

/* ── Animated Background ── */
.stApp {
    background: linear-gradient(135deg, var(--bg-dark) 0%, #0a0a2e 30%, #10103a 60%, #0d0d30 80%, var(--bg-dark) 100%);
    background-size: 400% 400%;
    animation: drift 25s ease infinite;
    color: var(--text-primary);
    font-family: 'Inter', sans-serif;
}
@keyframes drift {
    0%   { background-position: 0% 50%; }
    50%  { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

/* ── Hide defaults ── */
#MainMenu, footer, header { visibility: hidden; }

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: linear-gradient(180deg, var(--accent-1), var(--accent-2)); border-radius: 10px; }

/* ── Hero ── */
.hero { text-align: center; padding: 2rem 1rem 0.5rem; }
.hero-badge {
    display: inline-block;
    background: rgba(124,58,237,0.12);
    border: 1px solid rgba(124,58,237,0.25);
    color: #a78bfa;
    padding: 0.25rem 1rem;
    border-radius: 20px;
    font-size: 0.7rem;
    font-weight: 700;
    letter-spacing: 2.5px;
    text-transform: uppercase;
    margin-bottom: 0.4rem;
}
.hero-title {
    font-family: 'Space Grotesk', sans-serif;
    font-size: 3.6rem;
    font-weight: 700;
    background: linear-gradient(135deg, var(--accent-1), var(--accent-2), var(--accent-3), var(--accent-1));
    background-size: 300% 300%;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    animation: shimmer 6s ease infinite;
    letter-spacing: -2px;
    line-height: 1.1;
}
@keyframes shimmer {
    0%   { background-position: 0% 50%; }
    50%  { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}
.hero-sub {
    color: var(--text-secondary);
    font-size: 0.95rem;
    max-width: 580px;
    margin: 0.4rem auto 0;
}

/* ── Divider ── */
.ndiv {
    height: 1px;
    background: linear-gradient(90deg, transparent 5%, rgba(124,58,237,0.3) 35%, rgba(6,214,160,0.2) 65%, transparent 95%);
    margin: 1.2rem 0;
}

/* ── Glass Card ── */
.gl {
    background: var(--bg-card);
    backdrop-filter: blur(14px);
    -webkit-backdrop-filter: blur(14px);
    border: 1px solid var(--border);
    border-radius: 16px;
    padding: 1.4rem;
    margin-bottom: 1rem;
    position: relative;
    overflow: hidden;
    transition: all 0.35s cubic-bezier(0.4,0,0.2,1);
}
.gl::before {
    content: '';
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 1px;
    background: linear-gradient(90deg, transparent, rgba(124,58,237,0.35), rgba(6,214,160,0.25), transparent);
}
.gl:hover {
    border-color: rgba(124,58,237,0.18);
    box-shadow: 0 8px 35px rgba(124,58,237,0.07);
    transform: translateY(-1px);
}

/* ── Neon Result Card ── */
.neon {
    background: rgba(255,255,255,0.04);
    backdrop-filter: blur(18px);
    border: 1px solid rgba(124,58,237,0.2);
    border-radius: 20px;
    padding: 2rem;
    text-align: center;
    overflow: hidden;
    animation: neonIn 0.6s ease-out;
}
@keyframes neonIn {
    from { opacity: 0; transform: translateY(25px) scale(0.96); }
    to   { opacity: 1; transform: translateY(0) scale(1); }
}
.emo-icon { font-size: 5rem; animation: pop 0.8s ease; }
@keyframes pop {
    0%   { transform: scale(0); }
    60%  { transform: scale(1.25); }
    100% { transform: scale(1); }
}
.emo-lbl {
    font-family: 'Space Grotesk', sans-serif;
    font-size: 2.2rem;
    font-weight: 700;
    margin: 0.3rem 0 0.15rem;
}
.emo-conf { font-size: 1rem; color: var(--text-secondary); }

/* ── Metric Cards ── */
.mg { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 0.7rem; margin: 0.8rem 0; }
.mc {
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: 14px;
    padding: 1.1rem;
    text-align: center;
    transition: all 0.25s;
    position: relative;
    overflow: hidden;
}
.mc::after {
    content: '';
    position: absolute;
    bottom: 0; left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg, var(--accent-1), var(--accent-2));
    opacity: 0;
    transition: opacity 0.3s;
}
.mc:hover { transform: scale(1.02); }
.mc:hover::after { opacity: 1; }
.mv {
    font-family: 'Space Grotesk', sans-serif;
    font-size: 1.7rem;
    font-weight: 700;
    background: linear-gradient(135deg, var(--accent-1), var(--accent-2));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.ml {
    font-size: 0.65rem;
    color: var(--text-secondary);
    text-transform: uppercase;
    letter-spacing: 1.5px;
    margin-top: 0.25rem;
}

/* ── Section Header ── */
.sh {
    font-family: 'Space Grotesk', sans-serif;
    font-size: 1.15rem;
    font-weight: 600;
    color: #c8d0e0;
    margin: 1.3rem 0 0.7rem;
    padding-bottom: 0.4rem;
    border-bottom: 1px solid rgba(124,58,237,0.18);
    display: flex;
    align-items: center;
    gap: 0.5rem;
}
.sh-bar {
    width: 3px;
    height: 18px;
    background: linear-gradient(180deg, var(--accent-1), var(--accent-2));
    border-radius: 2px;
    flex-shrink: 0;
}

/* ── Tag ── */
.tag {
    display: inline-block;
    background: rgba(124,58,237,0.1);
    border: 1px solid rgba(124,58,237,0.2);
    color: #a78bfa;
    padding: 0.2rem 0.65rem;
    border-radius: 8px;
    font-size: 0.72rem;
    font-weight: 600;
    font-family: 'JetBrains Mono', monospace;
}

/* ── Feature Row ── */
.fr {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 0.55rem 0;
    border-bottom: 1px solid rgba(255,255,255,0.035);
}
.fn { color: #9ca3af; font-size: 0.82rem; }
.fv {
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.82rem;
    color: var(--accent-2);
    font-weight: 600;
}

/* ── Tabs ── */
.stTabs [data-baseweb="tab-list"] {
    gap: 3px;
    background: rgba(255,255,255,0.02);
    border: 1px solid rgba(255,255,255,0.04);
    border-radius: 14px;
    padding: 4px;
}
.stTabs [data-baseweb="tab"] {
    border-radius: 10px;
    color: var(--text-secondary);
    font-weight: 500;
    font-size: 0.85rem;
    padding: 8px 14px;
    transition: all 0.3s;
}
.stTabs [data-baseweb="tab"]:hover {
    color: #a78bfa;
    background: rgba(124,58,237,0.06);
}
.stTabs [aria-selected="true"] {
    background: linear-gradient(135deg, rgba(124,58,237,0.18), rgba(6,214,160,0.08)) !important;
    color: #fff !important;
    font-weight: 600;
}

/* ── Buttons ── */
.stButton > button {
    background: linear-gradient(135deg, var(--accent-1), var(--accent-2)) !important;
    color: #fff !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 0.7rem 2.2rem !important;
    font-weight: 700 !important;
    font-size: 0.95rem !important;
    letter-spacing: 0.4px !important;
    box-shadow: 0 4px 18px rgba(124,58,237,0.22) !important;
    transition: all 0.35s cubic-bezier(0.4,0,0.2,1) !important;
}
.stButton > button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 28px rgba(124,58,237,0.35) !important;
}

/* ── Inputs ── */
.stSelectbox > div > div,
.stNumberInput > div > div > input {
    background: rgba(255,255,255,0.04) !important;
    border-color: rgba(255,255,255,0.07) !important;
    color: var(--text-primary) !important;
    border-radius: 10px !important;
}

/* ── Footer ── */
.foot {
    text-align: center;
    padding: 1.8rem 0 0.8rem;
    color: #374151;
    font-size: 0.72rem;
    border-top: 1px solid rgba(255,255,255,0.03);
    margin-top: 2.5rem;
}
.foot b {
    background: linear-gradient(135deg, var(--accent-1), var(--accent-2));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
</style>
""", unsafe_allow_html=True)


# ============================================================================
# EMOTION CONFIG
# ============================================================================

EMO = {
    "Happiness": {"emoji": "😊", "c": "#fbbf24", "g": "rgba(251,191,36,0.25)"},
    "Sadness":   {"emoji": "😢", "c": "#3b82f6", "g": "rgba(59,130,246,0.25)"},
    "Anger":     {"emoji": "😠", "c": "#ef4444", "g": "rgba(239,68,68,0.25)"},
    "Anxiety":   {"emoji": "😰", "c": "#f97316", "g": "rgba(249,115,22,0.25)"},
    "Boredom":   {"emoji": "😐", "c": "#6b7280", "g": "rgba(107,114,128,0.25)"},
    "Neutral":   {"emoji": "😶", "c": "#06d6a0", "g": "rgba(6,214,160,0.25)"},
}
PLATFORMS = ["Instagram", "Twitter", "Facebook", "LinkedIn", "Snapchat", "Telegram", "Whatsapp"]
GENDERS = ["Female", "Male", "Non-binary"]


# ============================================================================
# LOAD ARTIFACTS
# ============================================================================

@st.cache_resource
def load_artifacts():
    """Load pipeline artifacts.  Keras uses lazy TF import."""
    with open("scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    with open("encoder.pkl", "rb") as f:
        encoder = pickle.load(f)
    with open("pipeline_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    dt = meta.get("model_deployment_type", "")

    if dt == "Keras":
        global TF_AVAILABLE
        try:
            import tensorflow as tf
            TF_AVAILABLE = True
        except ImportError:
            st.error("❌ Champion is Keras but tensorflow is not installed.")
            st.stop()
        km = tf.keras.models.load_model("champion_model.keras")

        class _KW:
            def predict_proba(self, X):
                return km.predict(X, verbose=0)
            def predict(self, X):
                return np.argmax(self.predict_proba(X), axis=1)

        mdl = _KW()
    else:
        with open("champion_model.pkl", "rb") as f:
            mdl = pickle.load(f)

    tm = None
    try:
        with open("best_tree_model.pkl", "rb") as f:
            tm = pickle.load(f)
    except FileNotFoundError:
        pass

    return mdl, scaler, encoder, meta, tm


try:
    model, scaler, encoder, META, tree_model = load_artifacts()
    FEATURES    = list(META.get("retained_features_list", []))
    CLASSES     = list(META.get("label_encoder_classes", []))
    DEPLOY_TYPE = META.get("model_deployment_type", "Unknown")
    N_CLASSES   = META.get("n_classes", len(CLASSES))
except Exception as e:
    st.error(f"❌ Artifact loading failed: {e}")
    st.info("Upload champion_model.pkl/.keras, scaler.pkl, encoder.pkl, pipeline_metadata.pkl.")
    st.stop()


# ============================================================================
# HELPERS
# ============================================================================

EPS = 1e-5


def _predict(X_scaled):
    """Return probability vector (1-D)."""
    return model.predict_proba(X_scaled)[0]


def _build(age, gender, platform, usage, posts, likes, comments, messages):
    """Engineer features → one-hot → scale.  Returns (scaled_array, raw_dict)."""
    raw = {
        "Age": age,
        "Daily_Usage_Time (minutes)": usage,
        "Posts_Per_Day": posts,
        "Likes_Received_Per_Day": likes,
        "Comments_Received_Per_Day": comments,
        "Messages_Sent_Per_Day": messages,
        "Interaction_Density": (likes + comments) / (usage + EPS),
        "Social_Velocity": likes / (posts + EPS),
        "Conversational_Reciprocity": messages / (comments + EPS),
        "Attention_Index": usage / (posts + EPS),
        "Engagement_Ratio": (likes + comments + messages) / (usage + EPS),
        "Content_Efficiency": likes / (usage * posts + EPS),
    }
    row = pd.DataFrame(columns=FEATURES, data=[np.zeros(len(FEATURES))])
    for c, v in raw.items():
        if c in row.columns:
            row[c] = v
    gc = f"Gender_{gender}"
    pc = f"Platform_{platform}"
    if gc in row.columns:
        row[gc] = 1.0
    if pc in row.columns:
        row[pc] = 1.0
    return scaler.transform(row), raw


def _layout(h=380, **kw):
    """Dark Plotly layout defaults."""
    d = dict(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#c8d0e0", family="Inter"),
        margin=dict(l=20, r=20, t=25, b=20),
        height=h,
    )
    d.update(kw)
    return d


def _emo(name):
    """Safe emotion config lookup."""
    return EMO.get(name, {"emoji": "🔮", "c": "#7c3aed", "g": "rgba(124,58,237,0.25)"})


def _sec(title):
    """Render a section header."""
    st.markdown(f'<div class="sh"><div class="sh-bar"></div>{title}</div>', unsafe_allow_html=True)


# ============================================================================
# HEADER
# ============================================================================

st.markdown("""
<div class="hero">
    <div class="hero-badge">AI · ML · Deep Learning</div>
    <div class="hero-title">🧠 NeuroSense</div>
    <div class="hero-sub">Predict dominant emotional states from social media behaviour using ensemble ML &amp; neural networks</div>
</div>
<div class="ndiv"></div>
""", unsafe_allow_html=True)


# ============================================================================
# 6  TABS
# ============================================================================

t1, t2, t3, t4, t5, t6 = st.tabs([
    "🔮 Predict", "📁 Batch", "📊 Performance",
    "🔬 Feature Lab", "🎛️ What-If", "ℹ️ About",
])


# ═══════════════════════════════════════════════════════════════════════════
# TAB 1 — PREDICT
# ═══════════════════════════════════════════════════════════════════════════
with t1:
    _sec("👤  User Profile & Engagement Metrics")
    c1, c2 = st.columns(2, gap="large")
    with c1:
        st.markdown('<div class="gl">', unsafe_allow_html=True)
        st.markdown("##### Demographics")
        p_age  = st.slider("Age", 10, 90, 25, key="p_age")
        p_gen  = st.selectbox("Gender", GENDERS, key="p_gen")
        p_plat = st.selectbox("Platform", PLATFORMS, key="p_plat")
        st.markdown("</div>", unsafe_allow_html=True)
    with c2:
        st.markdown('<div class="gl">', unsafe_allow_html=True)
        st.markdown("##### Daily Engagement")
        p_use = st.number_input("Usage Time (min)", 1, 1440, 120, key="p_use")
        p_pos = st.number_input("Posts", 0, 100, 3, key="p_pos")
        p_lik = st.number_input("Likes Received", 0, 10000, 45, key="p_lik")
        p_com = st.number_input("Comments Received", 0, 5000, 10, key="p_com")
        p_msg = st.number_input("Messages Sent", 0, 5000, 15, key="p_msg")
        st.markdown("</div>", unsafe_allow_html=True)

    st.markdown('<div class="ndiv"></div>', unsafe_allow_html=True)
    _, btn_col, _ = st.columns([1, 1, 1])
    with btn_col:
        go_pred = st.button("🚀  Analyze Profile", use_container_width=True, key="go_pred")

    if go_pred:
        xs, raw = _build(p_age, p_gen, p_plat, p_use, p_pos, p_lik, p_com, p_msg)
        probs   = _predict(xs)
        idx     = int(np.argmax(probs))
        emo_n   = CLASSES[idx]
        conf    = float(probs[idx])
        ec      = _emo(emo_n)

        # ── result card ──
        st.markdown(f"""
        <div class="neon" style="border-color:{ec['c']}40; box-shadow:0 0 50px {ec['g']};">
            <div class="emo-icon">{ec['emoji']}</div>
            <div class="emo-lbl" style="color:{ec['c']};">{emo_n}</div>
            <div class="emo-conf">Confidence: <strong>{conf:.1%}</strong></div>
        </div>
        """, unsafe_allow_html=True)

        # ── radar + bar ──
        r1, r2 = st.columns(2, gap="large")
        with r1:
            _sec("🎯 Probability Radar")
            rfig = go.Figure(go.Scatterpolar(
                r=list(probs) + [probs[0]],
                theta=CLASSES + [CLASSES[0]],
                fill="toself",
                fillcolor="rgba(124,58,237,0.1)",
                line=dict(color="#7c3aed", width=2.5),
                marker=dict(size=6, color="#06d6a0"),
            ))
            rfig.update_layout(**_layout(
                polar=dict(
                    bgcolor="rgba(0,0,0,0)",
                    radialaxis=dict(visible=True, range=[0, 1],
                                    gridcolor="rgba(255,255,255,0.05)",
                                    tickfont=dict(size=9, color="#4b5563")),
                    angularaxis=dict(gridcolor="rgba(255,255,255,0.04)",
                                     tickfont=dict(size=10, color="#9ca3af")),
                ), showlegend=False,
            ))
            st.plotly_chart(rfig, use_container_width=True)

        with r2:
            _sec("📊 Class Probabilities")
            bdf = pd.DataFrame({"Emotion": CLASSES, "P": probs}).sort_values("P")
            bcolors = [_emo(e)["c"] for e in bdf["Emotion"]]
            bfig = go.Figure(go.Bar(
                x=bdf["P"], y=bdf["Emotion"], orientation="h",
                marker=dict(color=bcolors),
                text=[f"{p:.1%}" for p in bdf["P"]],
                textposition="outside",
                textfont=dict(color="#9ca3af", size=11),
            ))
            bfig.update_layout(**_layout(
                xaxis=dict(range=[0, 1], gridcolor="rgba(255,255,255,0.03)",
                           tickformat=".0%", tickfont=dict(color="#4b5563")),
                yaxis=dict(tickfont=dict(color="#c8d0e0", size=11)),
            ))
            st.plotly_chart(bfig, use_container_width=True)

        # ── gauge ──
        _sec("⚡ Confidence Gauge")
        gfig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=conf * 100,
            number=dict(suffix="%", font=dict(size=38, color="#e0e6ed")),
            gauge=dict(
                axis=dict(range=[0, 100], tickcolor="#4b5563"),
                bar=dict(color=ec["c"]),
                bgcolor="rgba(255,255,255,0.02)",
                bordercolor="rgba(255,255,255,0.05)",
                steps=[
                    dict(range=[0, 33],  color="rgba(239,68,68,0.08)"),
                    dict(range=[33, 66], color="rgba(249,115,22,0.08)"),
                    dict(range=[66, 100], color="rgba(6,214,160,0.08)"),
                ],
                threshold=dict(line=dict(color="#06d6a0", width=3),
                               thickness=0.8, value=conf * 100),
            ),
        ))
        gfig.update_layout(**_layout(h=240))
        st.plotly_chart(gfig, use_container_width=True)

        # ── engineered features ──
        _sec("🧬 Engineered Features")
        eng_names = [
            ("Interaction Density", "Interaction_Density"),
            ("Social Velocity", "Social_Velocity"),
            ("Conversational Reciprocity", "Conversational_Reciprocity"),
            ("Attention Index", "Attention_Index"),
            ("Engagement Ratio", "Engagement_Ratio"),
            ("Content Efficiency", "Content_Efficiency"),
        ]
        fhtml = ""
        for label, key in eng_names:
            fhtml += f'<div class="fr"><span class="fn">{label}</span><span class="fv">{raw.get(key, 0):.4f}</span></div>'
        st.markdown(f'<div class="gl">{fhtml}</div>', unsafe_allow_html=True)

        # ── SHAP ──
        if SHAP_AVAILABLE and tree_model is not None:
            _sec("🔍 SHAP Feature Explanation")
            try:
                import matplotlib.pyplot as plt
                idf = pd.DataFrame(xs, columns=FEATURES)
                expl = shap.TreeExplainer(tree_model)
                sv   = expl(idf)
                if hasattr(sv, "values") and sv.values.ndim == 3:
                    bv = sv.base_values[0]
                    if isinstance(bv, (list, np.ndarray)):
                        bv = bv[idx]
                    sv0 = shap.Explanation(
                        values=sv.values[0, :, idx],
                        base_values=bv,
                        data=sv.data[0],
                        feature_names=FEATURES,
                    )
                else:
                    sv0 = sv[0]
                shap.plots.waterfall(sv0, show=False)
                st.pyplot(plt.gcf())
                plt.close("all")
            except Exception as ex:
                st.caption(f"SHAP unavailable: {str(ex)[:120]}")

        # ── all-class sunburst ──
        _sec("🌐 Probability Sunburst")
        sb_df = pd.DataFrame({"Emotion": CLASSES, "Probability": probs})
        sb_fig = px.sunburst(
            sb_df, path=["Emotion"], values="Probability",
            color="Probability",
            color_continuous_scale=[[0, "#7c3aed"], [1, "#06d6a0"]],
        )
        sb_fig.update_layout(**_layout(h=350))
        sb_fig.update_traces(textfont=dict(color="white"))
        st.plotly_chart(sb_fig, use_container_width=True)


# ═══════════════════════════════════════════════════════════════════════════
# TAB 2 — BATCH ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════
with t2:
    _sec("📁 Batch Prediction — CSV Upload")

    st.markdown("""
    <div class="gl">
        <p style="margin:0; color:#9ca3af;">Upload a CSV with columns:
        <code style="color:#06d6a0;">Age, Gender, Platform, Daily_Usage_Time (minutes),
        Posts_Per_Day, Likes_Received_Per_Day, Comments_Received_Per_Day,
        Messages_Sent_Per_Day</code></p>
    </div>
    """, unsafe_allow_html=True)

    up = st.file_uploader("Choose CSV", type=["csv"], key="batch_up")

    if up is not None:
        try:
            bdf = pd.read_csv(up)
            st.markdown(
                f'<span class="tag">{len(bdf)} rows × {len(bdf.columns)} cols</span>',
                unsafe_allow_html=True,
            )
            with st.expander("📋 Preview", expanded=True):
                st.dataframe(bdf.head(10), use_container_width=True)

            if st.button("🚀  Run Batch", key="batch_go"):
                res = []
                bar = st.progress(0, text="Predicting…")
                for i, (_, row) in enumerate(bdf.iterrows()):
                    try:
                        xs, _ = _build(
                            row.get("Age", 25),
                            row.get("Gender", "Male"),
                            row.get("Platform", "Instagram"),
                            row.get("Daily_Usage_Time (minutes)", 60),
                            row.get("Posts_Per_Day", 2),
                            row.get("Likes_Received_Per_Day", 20),
                            row.get("Comments_Received_Per_Day", 5),
                            row.get("Messages_Sent_Per_Day", 10),
                        )
                        pr = _predict(xs)
                        pi = int(np.argmax(pr))
                        res.append({"Predicted_Emotion": CLASSES[pi],
                                    "Confidence": float(pr[pi])})
                    except Exception:
                        res.append({"Predicted_Emotion": "Error", "Confidence": 0.0})
                    bar.progress((i + 1) / len(bdf))
                bar.empty()

                rdf = pd.concat([bdf, pd.DataFrame(res)], axis=1)
                _sec("📊 Results")
                st.dataframe(rdf, use_container_width=True)

                sc1, sc2 = st.columns(2)
                with sc1:
                    ed = rdf["Predicted_Emotion"].value_counts()
                    pfig = px.pie(
                        values=ed.values, names=ed.index, color=ed.index,
                        color_discrete_map={e: v["c"] for e, v in EMO.items()},
                        title="Distribution",
                    )
                    pfig.update_layout(**_layout(h=340))
                    st.plotly_chart(pfig, use_container_width=True)
                with sc2:
                    ac = rdf["Confidence"].mean()
                    top_emo = ed.index[0] if len(ed) > 0 else "N/A"
                    st.markdown(f"""
                    <div class="gl" style="text-align:center; padding:2rem;">
                        <div class="mv">{ac:.1%}</div><div class="ml">Avg Confidence</div>
                        <div style="margin-top:1.2rem;"><div class="mv">{len(rdf)}</div><div class="ml">Total Rows</div></div>
                        <div style="margin-top:1.2rem;"><div class="mv">{top_emo}</div><div class="ml">Most Frequent</div></div>
                    </div>
                    """, unsafe_allow_html=True)

                csv_out = rdf.to_csv(index=False)
                st.download_button("⬇️  Download CSV", csv_out,
                                   "neurosense_predictions.csv", "text/csv")

        except Exception as ex:
            st.error(f"Error: {ex}")


# ═══════════════════════════════════════════════════════════════════════════
# TAB 3 — MODEL PERFORMANCE
# ═══════════════════════════════════════════════════════════════════════════
with t3:
    champ = META.get("champion_model_name", "Unknown")
    perf  = META.get("model_performance", [])

    # champion banner
    st.markdown(f"""
    <div class="neon">
        <div style="font-size:0.65rem; color:#6b7280; text-transform:uppercase; letter-spacing:2.5px;">Champion</div>
        <div style="font-family:'Space Grotesk'; font-size:2rem; font-weight:700;
             background:linear-gradient(135deg,#7c3aed,#06d6a0);
             -webkit-background-clip:text; -webkit-text-fill-color:transparent; margin:0.3rem 0;">
            🏆 {champ}
        </div>
        <span class="tag">{DEPLOY_TYPE}</span>
    </div>
    """, unsafe_allow_html=True)

    # metric cards
    if perf:
        cp = perf[0]
        keys = [
            ("Accuracy", "Accuracy"), ("Precision (W)", "Precision W"),
            ("Recall (W)", "Recall W"), ("F1-Score (W)", "F1 Weighted"),
            ("Precision (Macro)", "Precision M"), ("F1-Score (Macro)", "F1 Macro"),
        ]
        cards = ""
        for k, label in keys:
            v = cp.get(k, 0)
            cards += f'<div class="mc"><div class="mv">{v:.4f}</div><div class="ml">{label}</div></div>'
        st.markdown(f'<div class="mg">{cards}</div>', unsafe_allow_html=True)

    # all models table
    if perf:
        _sec("📈 All Models Comparison")
        pdf = pd.DataFrame(perf)
        num_cols = [c for c in pdf.columns if c != "Model"]
        st.dataframe(
            pdf.style
            .format({c: "{:.4f}" for c in num_cols})
            .highlight_max(subset=num_cols, color="rgba(6,214,160,0.12)"),
            use_container_width=True,
        )

        # F1 bar
        _sec("📊 F1-Score (Weighted) Ranking")
        f1col = "F1-Score (W)"
        if f1col in pdf.columns:
            ps = pdf.sort_values(f1col, ascending=True)
            ff = go.Figure(go.Bar(
                x=ps[f1col], y=ps["Model"], orientation="h",
                marker=dict(color=ps[f1col],
                            colorscale=[[0, "#7c3aed"], [1, "#06d6a0"]]),
                text=[f"{v:.4f}" for v in ps[f1col]],
                textposition="outside",
                textfont=dict(color="#9ca3af", size=11),
            ))
            ff.update_layout(**_layout(
                h=max(250, len(ps) * 42),
                xaxis=dict(gridcolor="rgba(255,255,255,0.03)",
                           tickfont=dict(color="#4b5563")),
                yaxis=dict(tickfont=dict(color="#c8d0e0", size=11)),
            ))
            st.plotly_chart(ff, use_container_width=True)

    # feature importance + confusion matrix
    ic1, ic2 = st.columns(2, gap="large")

    fimp = META.get("feature_importance", {})
    with ic1:
        if fimp:
            _sec("🎯 Feature Importance")
            idf = (pd.DataFrame({"F": list(fimp.keys()), "I": list(fimp.values())})
                     .sort_values("I", ascending=True).tail(15))
            ifig = go.Figure(go.Bar(
                x=idf["I"], y=idf["F"], orientation="h",
                marker=dict(color=idf["I"],
                            colorscale=[[0, "#7c3aed"], [1, "#06d6a0"]]),
                text=[f"{v:.4f}" for v in idf["I"]],
                textposition="outside",
                textfont=dict(color="#9ca3af", size=10),
            ))
            ifig.update_layout(**_layout(h=400,
                xaxis=dict(gridcolor="rgba(255,255,255,0.03)",
                           tickfont=dict(color="#4b5563")),
                yaxis=dict(tickfont=dict(color="#c8d0e0", size=10)),
            ))
            st.plotly_chart(ifig, use_container_width=True)

    cm_raw = META.get("confusion_matrix")
    with ic2:
        if cm_raw is not None:
            _sec("🗺️ Confusion Matrix")
            cma = np.array(cm_raw, dtype=float)
            row_sums = cma.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1          # prevent /0
            cmn = cma / row_sums
            cfig = go.Figure(go.Heatmap(
                z=cmn, x=CLASSES, y=CLASSES,
                colorscale=[[0, "#05051a"], [0.5, "#7c3aed"], [1, "#06d6a0"]],
                text=[[f"{v:.2f}" for v in r] for r in cmn],
                texttemplate="%{text}",
                textfont=dict(size=11, color="white"),
            ))
            cfig.update_layout(**_layout(h=400,
                xaxis=dict(title="Predicted", tickfont=dict(color="#c8d0e0")),
                yaxis=dict(title="True", tickfont=dict(color="#c8d0e0"),
                           autorange="reversed"),
            ))
            st.plotly_chart(cfig, use_container_width=True)

    # class distribution
    cdist = META.get("class_distribution", {})
    if cdist:
        _sec("📦 Training Class Distribution")
        try:
            dist_labels = [CLASSES[int(k)] if int(k) < len(CLASSES) else str(k)
                           for k in cdist.keys()]
        except (ValueError, IndexError):
            dist_labels = list(cdist.keys())
        dist_vals = list(cdist.values())
        dc = [_emo(l)["c"] for l in dist_labels]
        dfig = go.Figure(go.Bar(
            x=dist_labels, y=dist_vals,
            marker=dict(color=dc),
            text=dist_vals, textposition="outside",
            textfont=dict(color="#9ca3af"),
        ))
        dfig.update_layout(**_layout(h=280,
            xaxis=dict(tickfont=dict(color="#c8d0e0")),
            yaxis=dict(gridcolor="rgba(255,255,255,0.03)",
                       tickfont=dict(color="#4b5563")),
        ))
        st.plotly_chart(dfig, use_container_width=True)


# ═══════════════════════════════════════════════════════════════════════════
# TAB 4 — FEATURE LAB
# ═══════════════════════════════════════════════════════════════════════════
with t4:
    _sec("🧬 Engineered Features — Explained")
    st.markdown("""
    <div class="gl"><p style="color:#9ca3af; margin:0;">
    The pipeline derives <strong style="color:#06d6a0;">6 behavioural interaction features</strong>
    from raw social media metrics.  These capture engagement patterns that raw counts alone cannot express.
    </p></div>
    """, unsafe_allow_html=True)

    eng_cards = [
        ("Interaction Density", "(Likes + Comments) / Usage", "Engagement intensity per minute of screen time"),
        ("Social Velocity", "Likes / Posts", "Average likes per content piece — content quality signal"),
        ("Conversational Reciprocity", "Messages / Comments", "Outgoing-to-incoming ratio — social balance"),
        ("Attention Index", "Usage / Posts", "Time per post — passive consumers score high"),
        ("Engagement Ratio", "(L + C + M) / Usage", "Holistic engagement across all interaction types"),
        ("Content Efficiency", "Likes / (Usage × Posts)", "Per-post, per-minute effectiveness — content ROI"),
    ]
    for name, formula, desc in eng_cards:
        st.markdown(f"""
        <div class="gl" style="padding:1rem 1.4rem;">
            <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:0.5rem;">
                <div>
                    <div style="color:#e0e6ed; font-weight:600;">{name}</div>
                    <div style="color:#6b7280; font-size:0.78rem; margin-top:0.15rem;">{desc}</div>
                </div>
                <span class="tag" style="white-space:nowrap;">{formula}</span>
            </div>
        </div>
        """, unsafe_allow_html=True)

    # importance ranking
    if fimp:
        _sec("🏅 Feature Importance Ranking")
        ranked = sorted(fimp.items(), key=lambda x: x[1], reverse=True)
        max_val = ranked[0][1] if ranked else 1
        rhtml = ""
        for i, (fn, fv) in enumerate(ranked):
            pct = (fv / max_val * 100) if max_val > 0 else 0
            rhtml += f"""
            <div style="display:flex; align-items:center; gap:0.7rem; padding:0.45rem 0; border-bottom:1px solid rgba(255,255,255,0.03);">
                <div style="color:#4b5563; font-size:0.75rem; width:22px; text-align:right;">#{i+1}</div>
                <div style="flex:1;">
                    <div style="color:#c8d0e0; font-size:0.82rem;">{fn}</div>
                    <div style="background:rgba(255,255,255,0.03); border-radius:3px; height:5px; margin-top:3px; overflow:hidden;">
                        <div style="width:{pct}%; height:100%; background:linear-gradient(90deg,#7c3aed,#06d6a0); border-radius:3px;"></div>
                    </div>
                </div>
                <span class="tag">{fv:.5f}</span>
            </div>"""
        st.markdown(f'<div class="gl">{rhtml}</div>', unsafe_allow_html=True)

    # feature set tags
    _sec("📋 Full Feature Set")
    cols = st.columns(3)
    for i, fn in enumerate(FEATURES):
        cols[i % 3].markdown(f'<span class="tag" style="margin:2px;">{fn}</span>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════════════════════════
# TAB 5 — WHAT-IF SIMULATOR
# ═══════════════════════════════════════════════════════════════════════════
with t5:
    _sec("🎛️ What-If Sensitivity Simulator")
    st.markdown("""
    <div class="gl"><p style="color:#9ca3af; margin:0;">
    Adjust a <strong style="color:#06d6a0;">single feature</strong> while keeping others fixed.
    Watch how predicted probabilities shift — revealing which inputs the model is most sensitive to.
    </p></div>
    """, unsafe_allow_html=True)

    wl, wr = st.columns([1, 2], gap="large")
    with wl:
        st.markdown('<div class="gl">', unsafe_allow_html=True)
        st.markdown("##### ⚙️ Baseline")
        w_age  = st.slider("Age", 10, 90, 25, key="w_age")
        w_gen  = st.selectbox("Gender", GENDERS, key="w_gen")
        w_plat = st.selectbox("Platform", PLATFORMS, key="w_plat")
        w_use  = st.number_input("Usage (min)", 1, 1440, 120, key="w_use")
        w_pos  = st.number_input("Posts", 0, 100, 3, key="w_pos")
        w_lik  = st.number_input("Likes", 0, 10000, 45, key="w_lik")
        w_com  = st.number_input("Comments", 0, 5000, 10, key="w_com")
        w_msg  = st.number_input("Messages", 0, 5000, 15, key="w_msg")
        st.markdown("</div>", unsafe_allow_html=True)

    with wr:
        sweep_feat = st.selectbox("Feature to Sweep", [
            "Daily Usage Time", "Posts Per Day", "Likes Received",
            "Comments Received", "Messages Sent", "Age",
        ], key="sw_feat")

        sweep_cfg = {
            "Daily Usage Time":  ("usage", 1,  500, 25),
            "Posts Per Day":     ("posts", 0,   50, 3),
            "Likes Received":    ("likes", 0,  500, 25),
            "Comments Received": ("comments", 0, 200, 10),
            "Messages Sent":     ("messages", 0, 200, 10),
            "Age":               ("age",  10,  80, 5),
        }
        param, lo, hi, step = sweep_cfg[sweep_feat]
        vals = list(range(lo, hi + 1, step))

        traces = {cn: [] for cn in CLASSES}
        preds  = []

        base = dict(age=w_age, gender=w_gen, platform=w_plat,
                    usage=w_use, posts=w_pos, likes=w_lik,
                    comments=w_com, messages=w_msg)

        for v in vals:
            a = base.copy()
            a[param] = v
            xs, _ = _build(a["age"], a["gender"], a["platform"],
                           a["usage"], a["posts"], a["likes"],
                           a["comments"], a["messages"])
            pr = _predict(xs)
            for ci, cn in enumerate(CLASSES):
                traces[cn].append(float(pr[ci]))
            preds.append(CLASSES[int(np.argmax(pr))])

        # line chart
        _sec("📈 Probability Sweep")
        lfig = go.Figure()
        for cn in CLASSES:
            lfig.add_trace(go.Scatter(
                x=vals, y=traces[cn], mode="lines+markers",
                name=cn, line=dict(color=_emo(cn)["c"], width=2.5),
                marker=dict(size=5),
            ))
        lfig.update_layout(**_layout(h=400,
            xaxis=dict(title=sweep_feat,
                       gridcolor="rgba(255,255,255,0.03)",
                       tickfont=dict(color="#4b5563")),
            yaxis=dict(title="Probability", range=[0, 1],
                       gridcolor="rgba(255,255,255,0.03)",
                       tickfont=dict(color="#4b5563")),
            legend=dict(font=dict(color="#9ca3af", size=10)),
        ))
        st.plotly_chart(lfig, use_container_width=True)

        # emoji strip
        _sec("🔮 Predicted Emotion at Each Point")
        strip = '<div style="display:flex; gap:3px; flex-wrap:wrap;">'
        for sv, pe in zip(vals, preds):
            ec = _emo(pe)
            strip += (
                f'<div style="background:rgba(255,255,255,0.03); border:1px solid {ec["c"]}25;'
                f'border-radius:8px; padding:0.25rem 0.5rem; text-align:center; min-width:50px;">'
                f'<div style="font-size:1.1rem;">{ec["emoji"]}</div>'
                f'<div style="font-size:0.6rem; color:#6b7280;">{sv}</div></div>'
            )
        strip += "</div>"
        st.markdown(strip, unsafe_allow_html=True)

        # sensitivity summary
        _sec("📐 Sensitivity Summary")
        changes = {}
        for cn in CLASSES:
            arr = traces[cn]
            changes[cn] = max(arr) - min(arr) if arr else 0
        sens_sorted = sorted(changes.items(), key=lambda x: x[1], reverse=True)
        shtml = ""
        for cn, delta in sens_sorted:
            ec = _emo(cn)
            pct = delta * 100
            shtml += f"""
            <div class="fr">
                <span class="fn">{ec['emoji']} {cn}</span>
                <span class="fv" style="color:{ec['c']};">Δ {pct:.1f}%</span>
            </div>"""
        st.markdown(f'<div class="gl">{shtml}</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════════════════════════
# TAB 6 — ABOUT
# ═══════════════════════════════════════════════════════════════════════════
with t6:
    st.markdown(f"""
    <div class="neon" style="text-align:left;">
        <div style="text-align:center; margin-bottom:1rem;">
            <div style="font-family:'Space Grotesk'; font-size:2rem; font-weight:700;
                 background:linear-gradient(135deg,#7c3aed,#06d6a0);
                 -webkit-background-clip:text; -webkit-text-fill-color:transparent;">
                🧠 About NeuroSense
            </div>
        </div>
        <p style="color:#9ca3af;">NeuroSense predicts a user's dominant emotional state from social
        media engagement patterns.  It combines classical ML, gradient boosting ensembles, and deep
        neural networks in a fully automated training pipeline with production-grade deployment.</p>
    </div>
    """, unsafe_allow_html=True)

    a1, a2 = st.columns(2, gap="large")

    with a1:
        st.markdown("""
        <div class="gl">
            <div class="sh" style="margin-top:0;"><div class="sh-bar"></div>⚙️ Data Pipeline</div>
            <ul style="color:#9ca3af; padding-left:1.1rem; font-size:0.85rem; line-height:1.9;">
                <li>Multi-stage anomaly-resilient cleaning (7 anomaly types)</li>
                <li>6 engineered behavioural features</li>
                <li>One-hot encoding + strict schema alignment</li>
                <li>Multicollinearity filter (r &gt; 0.85)</li>
                <li>Train-only StandardScaler (leak-proof)</li>
                <li>Label typo correction ("Agression" → "Anger")</li>
            </ul>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("""
        <div class="gl">
            <div class="sh" style="margin-top:0;"><div class="sh-bar"></div>📊 Evaluation</div>
            <ul style="color:#9ca3af; padding-left:1.1rem; font-size:0.85rem; line-height:1.9;">
                <li>Weighted &amp; Macro F1, Precision, Recall</li>
                <li>OvR ROC-AUC &amp; PR-AUC Curves</li>
                <li>Per-class Classification Report</li>
                <li>SHAP TreeExplainer Importance</li>
                <li>Learning Curve Diagnostics</li>
                <li>Normalized Confusion Matrix</li>
            </ul>
        </div>
        """, unsafe_allow_html=True)

    with a2:
        st.markdown("""
        <div class="gl">
            <div class="sh" style="margin-top:0;"><div class="sh-bar"></div>🤖 Models (8 Total)</div>
            <ul style="color:#9ca3af; padding-left:1.1rem; font-size:0.85rem; line-height:1.9;">
                <li>📊 Logistic Regression (multinomial, balanced)</li>
                <li>🌲 Random Forest (300 trees, balanced)</li>
                <li>🚀 CatBoost (RandomizedSearchCV, 20 iter)</li>
                <li>⚡ LightGBM (RandomizedSearchCV, 20 iter)</li>
                <li>🎯 XGBoost (RandomizedSearchCV, 20 iter)</li>
                <li>🧠 MLP (256→128→64, BN + Dropout)</li>
                <li>🔮 Swish-Net (512→256→128→64, Swish)</li>
                <li>🏆 Soft-Vote Ensemble (top-3 avg)</li>
            </ul>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("""
        <div class="gl">
            <div class="sh" style="margin-top:0;"><div class="sh-bar"></div>🛠️ Tech Stack</div>
            <table style="width:100%; color:#9ca3af; font-size:0.85rem;">
                <tr><td style="padding:0.3rem 0;"><strong style="color:#c8d0e0;">ML / DL</strong></td>
                    <td>scikit-learn · CatBoost · LightGBM · XGBoost · TF/Keras</td></tr>
                <tr><td style="padding:0.3rem 0;"><strong style="color:#c8d0e0;">XAI</strong></td>
                    <td>SHAP (TreeExplainer)</td></tr>
                <tr><td style="padding:0.3rem 0;"><strong style="color:#c8d0e0;">Frontend</strong></td>
                    <td>Streamlit · Plotly · Custom CSS</td></tr>
                <tr><td style="padding:0.3rem 0;"><strong style="color:#c8d0e0;">Data</strong></td>
                    <td>Pandas · NumPy · SciPy</td></tr>
            </table>
        </div>
        """, unsafe_allow_html=True)

    # architecture flow
    _sec("🏗️ System Architecture")
    st.markdown("""
    <div class="gl" style="text-align:center; padding:1.8rem;">
        <div style="display:flex; justify-content:center; align-items:center; gap:0.8rem; flex-wrap:wrap;">
            <span class="tag" style="padding:0.4rem 0.9rem; font-size:0.8rem;">📁 Raw CSVs</span>
            <span style="color:#4b5563;">→</span>
            <span class="tag" style="padding:0.4rem 0.9rem; font-size:0.8rem;">🧹 Cleaning</span>
            <span style="color:#4b5563;">→</span>
            <span class="tag" style="padding:0.4rem 0.9rem; font-size:0.8rem;">⚙️ Feature Eng</span>
            <span style="color:#4b5563;">→</span>
            <span class="tag" style="padding:0.4rem 0.9rem; font-size:0.8rem;">📐 Scaling</span>
            <span style="color:#4b5563;">→</span>
            <span class="tag" style="padding:0.4rem 0.9rem; font-size:0.8rem; background:rgba(6,214,160,0.12); border-color:rgba(6,214,160,0.25); color:#06d6a0;">🏆 Champion</span>
            <span style="color:#4b5563;">→</span>
            <span class="tag" style="padding:0.4rem 0.9rem; font-size:0.8rem;">🔮 Predict</span>
        </div>
    </div>
    """, unsafe_allow_html=True)

    # deployment info
    _sec("🚀 Deployment Precautions")
    st.markdown(f"""
    <div class="gl">
        <div class="fr"><span class="fn">Champion Model</span><span class="fv">{champ}</span></div>
        <div class="fr"><span class="fn">Deployment Type</span><span class="fv">{DEPLOY_TYPE}</span></div>
        <div class="fr"><span class="fn">Pickle Contract</span><span class="fv">ChampionModelWrapper</span></div>
        <div class="fr"><span class="fn">TF Strategy</span><span class="fv">Lazy (only if Keras wins)</span></div>
        <div class="fr"><span class="fn">Python Version</span><span class="fv">3.10 (.python-version)</span></div>
        <div class="fr"><span class="fn">TF Package</span><span class="fv">tensorflow-cpu (memory-safe)</span></div>
        <div class="fr" style="border:none;"><span class="fn">Features</span><span class="fv">{len(FEATURES)} retained</span></div>
    </div>
    """, unsafe_allow_html=True)


# ============================================================================
# FOOTER
# ============================================================================

st.markdown("""
<div class="foot">
    Built with <b>NeuroSense</b> Analytics Engine &nbsp;•&nbsp;
    ML &amp; Deep Learning &nbsp;•&nbsp; Streamlit + Plotly
</div>
""", unsafe_allow_html=True)


Project Link:- https://sentinet-social-11.streamlit.app/